## NBA Quant AI — GPU Evolution v2 (Kaggle Edition)

**Adapted from nba_gpu_v2.ipynb for Kaggle P100/2xT4 runtime.**

| Optimization | Colab v2 | Kaggle v2 |
|---|---|---|
| Population | 24 | 30 |
| CV Folds | 2 | 2 |
| GPU | T4 15GB | P100 16GB or 2xT4 |
| Session limit | 4h | 9h (more gens!) |
| Feature cache | Google Drive | /kaggle/working/ |
| State save | Google Drive every gen | /kaggle/working/ every gen |
| Secrets | Colab userdata | Kaggle UserSecretsClient |
| Internet on GPU | YES | NO |

---

## IMPORTANT: Run cells 1-2 with Internet ON (Accelerator: None). Then switch to GPU and run cells 3-6.

**Why:** Kaggle disables internet access during GPU sessions. ALL downloads (pip install,
git clone, model checkpoint downloads) MUST happen in cells 1-2 while internet is enabled
(Accelerator: None). Once everything is cached on disk, switch to GPU and run cells 3-6.

**Steps:**
1. Settings -> Accelerator: **None** (internet ON by default)
2. Run **Cell 1** -- installs deps, pre-caches TabICL HF checkpoint, clones repo
3. Run **Cell 2** -- builds feature cache (~30 min) or loads from /kaggle/working/
4. Settings -> Accelerator: **GPU (P100 or 2xT4)** <- internet goes OFF here
5. Run **Cells 3-6** -- seed population, evolution loop, results, inject to HF islands

**Secrets:** Add in Kaggle notebook -> Settings -> Secrets:
- `HF_TOKEN` -- HuggingFace token
- `DATABASE_URL` -- Supabase pooler connection string

**Resuming across sessions:** Download `evolution_state_v2_kaggle.json` from Kaggle output.
Upload it to /kaggle/working/nba-quant-gpu/ before running Cell 4 in the next session.

**Target to beat:** 0.21837 (S13 CatBoost gen815)

In [ ]:
# ============================================================
# CELL 1: SETUP -- DEPS + DOWNLOADS (Internet ON, Accelerator: None)
# ============================================================
# MUST run BEFORE switching to GPU accelerator.
# Kaggle caches pip packages and HF model weights -- subsequent runs are faster.
import subprocess, sys, os, time, gc, json, warnings
warnings.filterwarnings('ignore')

# /kaggle/working/ persists within a session; files appear in Kaggle output.
# NOTE: /kaggle/working/ is NOT preserved across separate Kaggle sessions.
CACHE_DIR = '/kaggle/working/nba-quant-gpu'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

# Kaggle secrets -- add HF_TOKEN and DATABASE_URL in notebook Settings -> Secrets
from kaggle_secrets import UserSecretsClient
secret = UserSecretsClient()
for key in ['HF_TOKEN', 'DATABASE_URL']:
    try:
        v = secret.get_secret(key)
        if v:
            os.environ[key] = v
            print(f'{key}: OK')
    except Exception:
        print(f'{key}: not set -- add to Kaggle Secrets before continuing')

# Install dependencies -- MUST have internet ON (Accelerator: None)
t0 = time.time()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost', 'lightgbm', 'catboost', 'scikit-learn', 'pandas', 'numpy',
    'scipy', 'tabicl', 'huggingface_hub', 'psycopg2-binary', 'requests'])
print(f'Deps installed: {time.time()-t0:.0f}s')

# torch is pre-installed on Kaggle GPU images
import torch, numpy as np
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
# CUDA will be False here (CPU/no-accelerator mode) -- expected.

# Pre-cache TabICL checkpoint from HuggingFace Hub -- MUST do with internet ON.
# Kaggle preserves ~/.cache/huggingface/ within a session, so this download
# survives the accelerator switch that disables internet.
t0 = time.time()
try:
    from tabicl import TabICLClassifier
    _m = TabICLClassifier()
    _m.fit(np.random.randn(50, 5), np.random.randint(0, 2, 50))
    del _m
    gc.collect()
    print(f'TabICL checkpoint: downloaded & cached ({time.time()-t0:.0f}s)')
except Exception as e:
    print(f'TabICL cache FAILED: {e}')
    print('ERROR: Do NOT switch to GPU until TabICL is cached. Fix the error above.')

# Clone repo -- needed for feature engine and historical game data
REPO_DIR = '/kaggle/working/nomos-nba-agent'
if not os.path.exists(REPO_DIR):
    print('Cloning nomos-nba-agent (depth=1)...')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/LBJLincoln/nomos-nba-agent.git',
                    REPO_DIR], check=True)
    print(f'Repo cloned in {time.time()-t0:.0f}s')
else:
    print(f'Repo already at {REPO_DIR}')

print('\nCell 1 complete. Run Cell 2 to build feature cache, then switch to GPU.')

In [ ]:
# ============================================================
# CELL 2: BUILD OR LOAD FEATURES (cached in /kaggle/working/)
# ============================================================
# Can run with internet ON (CPU mode) -- GPU not needed for feature building.
# First run: ~30 min to build. Subsequent loads within session: ~1s.
# TIP: Upload a pre-built features_cache_v38.npz as a Kaggle Dataset to skip building.
import sys, os, json, time
import numpy as np
from pathlib import Path

CACHE_DIR = '/kaggle/working/nba-quant-gpu'
REPO_DIR  = '/kaggle/working/nomos-nba-agent'
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_FILE = Path(f'{CACHE_DIR}/features_cache_v38.npz')  # versioned by engine version

# If cache was pre-uploaded as a Kaggle Dataset (dataset slug: nba-quant-features),
# copy it into working dir to skip the 30-min build step.
DATASET_CACHE = Path('/kaggle/input/nba-quant-features/features_cache_v38.npz')
if not CACHE_FILE.exists() and DATASET_CACHE.exists():
    import shutil
    shutil.copy(str(DATASET_CACHE), str(CACHE_FILE))
    print(f'Copied cache from Kaggle Dataset: {DATASET_CACHE}')

if CACHE_FILE.exists():
    print('Loading cached features from /kaggle/working/...')
    t0 = time.time()
    cached = np.load(str(CACHE_FILE), allow_pickle=True)
    X_full = cached['X']
    y_full = cached['y']
    feature_names = cached['feature_names'].tolist()
    print(f'Loaded: {X_full.shape} in {time.time()-t0:.1f}s')
else:
    print('Building features (~30 min). Will cache to /kaggle/working/...')
    print('TIP: Download features_cache_v38.npz from output and upload as Kaggle Dataset')
    print('     to skip this step in future sessions.')
    t0 = time.time()

    if not os.path.exists(REPO_DIR):
        raise RuntimeError('Repo not found -- did Cell 1 complete successfully?')
    sys.path.insert(0, f'{REPO_DIR}/hf-space')

    data_dir = Path(f'{REPO_DIR}/hf-space/data/historical')
    games = []
    for f in sorted(data_dir.glob('games-*.json')):
        raw = json.loads(f.read_text())
        if isinstance(raw, list):
            games.extend(raw)
        elif isinstance(raw, dict) and 'games' in raw:
            games.extend(raw['games'])
    print(f'Games: {len(games)}')

    from features.engine import NBAFeatureEngine
    engine = NBAFeatureEngine()
    X_raw, y_raw, feature_names = engine.build(games)
    X_full = np.nan_to_num(np.array(X_raw, dtype=np.float32))  # float32 saves RAM
    y_full = np.array(y_raw, dtype=np.int32)
    variances = np.var(X_full, axis=0)
    valid = variances > 1e-10
    X_full = X_full[:, valid]
    feature_names = [f for f, v in zip(feature_names, valid) if v]
    np.savez_compressed(str(CACHE_FILE), X=X_full, y=y_full,
                        feature_names=np.array(feature_names))
    print(f'Built & cached: {X_full.shape} in {time.time()-t0:.0f}s')
    print(f'Cache file: {CACHE_FILE}')

# Subsample: last 6000 games (faster eval, most recent = most relevant)
SUBSAMPLE = 6000
if X_full.shape[0] > SUBSAMPLE:
    X = X_full[-SUBSAMPLE:]
    y = y_full[-SUBSAMPLE:]
    print(f'Subsampled: {X_full.shape[0]} -> {SUBSAMPLE} (last {SUBSAMPLE} games)')
else:
    X = X_full
    y = y_full

n_features = X.shape[1]
print(f'Ready: {X.shape} ({n_features} features)')
print('\nCell 2 complete. Now switch Accelerator to GPU, then run Cells 3-6.')

In [ ]:
# ============================================================
# CELL 3: SEED POPULATION FROM 6 ISLANDS + CONFIG
# ============================================================
# Run with GPU accelerator ON (internet is OFF on Kaggle GPU sessions).
# Island seed fetching will fail silently -- that is expected.
# Fall back to cached seeds (from CPU run) or random initialization.
import requests, random, signal, os, json, time, gc, warnings
import numpy as np
import torch
from pathlib import Path
from collections import Counter
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')

CACHE_DIR = '/kaggle/working/nba-quant-gpu'
REPO_DIR  = '/kaggle/working/nomos-nba-agent'
os.makedirs(CACHE_DIR, exist_ok=True)

# Reload features if kernel was restarted after switching accelerator
if 'X' not in dir() or 'feature_names' not in dir():
    print('Reloading feature cache (kernel restarted after GPU switch)...')
    import sys
    sys.path.insert(0, f'{REPO_DIR}/hf-space')
    cached = np.load(f'{CACHE_DIR}/features_cache_v38.npz', allow_pickle=True)
    X_full = cached['X']; y_full = cached['y']
    feature_names = cached['feature_names'].tolist()
    SUBSAMPLE = 6000
    X = X_full[-SUBSAMPLE:] if X_full.shape[0] > SUBSAMPLE else X_full
    y = y_full[-SUBSAMPLE:] if y_full.shape[0] > SUBSAMPLE else y_full
    n_features = X.shape[1]
    print(f'Reloaded: {X.shape}')

# GPU detection -- Kaggle offers P100 (16GB) or 2xT4
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} ({props.total_memory/1e9:.1f} GB)')
    gpu0_name = torch.cuda.get_device_properties(0).name
    if 'P100' in gpu0_name:
        print('Detected: Kaggle P100 (16GB) -- best single-GPU option')
    elif 'T4' in gpu0_name:
        print(f'Detected: Kaggle T4 x{n_gpus}')
else:
    print('WARNING: No CUDA. Did you switch to GPU accelerator?')

# ── PLATFORM CONFIG ──
# kaggle: 9h session, P100 16GB or 2xT4 -- more gens than Colab 4h free
PLATFORM = 'kaggle'  # options: 'colab_free', 'colab_pro', 'lightning', 'kaggle'

CONFIGS = {
    'colab_free': {'POP': 24, 'FOLDS': 2, 'GENS': 500,  'ELITE': 4, 'TIMEOUT': 90},
    'colab_pro':  {'POP': 40, 'FOLDS': 3, 'GENS': 500,  'ELITE': 6, 'TIMEOUT': 120},
    'lightning':  {'POP': 40, 'FOLDS': 3, 'GENS': 1000, 'ELITE': 6, 'TIMEOUT': 120},
    'kaggle':     {'POP': 30, 'FOLDS': 2, 'GENS': 500,  'ELITE': 5, 'TIMEOUT': 90},
}
cfg          = CONFIGS[PLATFORM]
POP_SIZE     = cfg['POP']
N_SPLITS     = cfg['FOLDS']
TOTAL_GENS   = cfg['GENS']
ELITE_SIZE   = cfg['ELITE']
EVAL_TIMEOUT = cfg['TIMEOUT']

# Evolution params
PURGE_GAP       = 5
TARGET_FEATURES = 60
MAX_FEATURES    = 200
MUTATION_RATE   = 0.10
CROSSOVER_RATE  = 0.80
MUT_FLOOR       = 0.05
MUT_DECAY       = 0.998

_xgb_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_WEIGHTS = {'tabicl': 50, 'xgboost': 15, 'catboost': 10, 'lightgbm': 10, 'extra_trees': 15}

tscv   = TimeSeriesSplit(n_splits=N_SPLITS)
splits = [(tr[:-PURGE_GAP] if len(tr) > PURGE_GAP + 50 else tr, te)
          for tr, te in tscv.split(X)]

print(f'\nPlatform: {PLATFORM} | Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS}')
print(f'XGBoost device: {_xgb_device}')
print(f'Session: 9h | Est: ~{3600 // max(POP_SIZE * 8, 1)} gens/hour')
print(f'Budget: ~{int(9 * 3600 / max(POP_SIZE * 8, 1))} gens possible in 9h')

# ── Fetch seeds from 6 islands ──
# Kaggle GPU sessions have no internet -- these requests fail silently.
# Seeds from a CPU-mode run of this cell are cached to disk and reloaded here.
ISLANDS = {
    'S10': 'https://nomos42-nba-quant.hf.space',
    'S11': 'https://nomos42-nba-quant-2.hf.space',
    'S12': 'https://nomos42-nba-evo-3.hf.space',
    'S13': 'https://nomos42-nba-evo-4.hf.space',
    'S14': 'https://nomos42-nba-evo-5.hf.space',
    'S15': 'https://nomos42-nba-evo-6.hf.space',
}

SEEDS_CACHE  = Path(f'{CACHE_DIR}/island_seeds_cache.json')
island_seeds = []

print('\nFetching island seeds (fails silently in GPU mode -- no internet)...')
for name, url in ISLANDS.items():
    try:
        resp = requests.get(f'{url}/api/results', timeout=5)
        if resp.status_code != 200:
            continue
        data = resp.json()
        best = data.get('best', {})
        island_seeds.append({
            'source': name, 'brier': best.get('brier', 1.0),
            'features': best.get('selected_features', []),
            'model_type': best.get('model_type', 'xgboost'),
        })
        print(f'  {name}: brier={best.get("brier","?"):.5f} '
              f'model={best.get("model_type","?")} feat={best.get("n_features","?")}')
        for i, ind in enumerate(data.get('top5', [])[:2]):
            island_seeds.append({
                'source': f'{name}_top{i+1}', 'brier': ind.get('brier', 1.0),
                'features': ind.get('selected_features', []),
                'model_type': ind.get('model_type', 'xgboost'),
            })
    except Exception as e:
        print(f'  {name}: {e} (expected in GPU mode)')

if island_seeds:
    SEEDS_CACHE.write_text(json.dumps(island_seeds))
    print(f'Seeds fetched: {len(island_seeds)} -- cached to {SEEDS_CACHE}')
elif SEEDS_CACHE.exists():
    island_seeds = json.loads(SEEDS_CACHE.read_text())
    print(f'Seeds loaded from disk cache: {len(island_seeds)}')
else:
    print('No seeds available (internet off + no cache) -- random initialization.')

In [ ]:
# ============================================================
# CELL 4: EVOLUTION ENGINE + POPULATION INIT
# ============================================================

def make_model(model_type, hp):
    if model_type == 'xgboost':
        return xgb.XGBClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            tree_method='hist', device=_xgb_device,
            random_state=42, verbosity=0)
    elif model_type == 'lightgbm':
        return lgb.LGBMClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            random_state=42, verbose=-1)
    elif model_type == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 8),
            min_samples_leaf=5, random_state=42, n_jobs=-1)
    elif model_type == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=min(hp.get('n_estimators', 200), 300),
            depth=min(hp.get('max_depth', 6), 10),
            learning_rate=hp.get('learning_rate', 0.05),
            random_state=42, verbose=0)
    elif model_type == 'tabicl':
        from tabicl import TabICLClassifier
        return TabICLClassifier()
    else:
        return ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)


class _Timeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise _Timeout()


def evaluate(features_mask, model_type, hp, timeout=EVAL_TIMEOUT):
    """Walk-forward evaluation with SIGALRM timeout protection."""
    indices = [i for i, v in enumerate(features_mask) if v]
    if len(indices) < 5:
        return 1.0
    if len(indices) > MAX_FEATURES:
        indices = indices[:MAX_FEATURES]
    X_sub = X[:, indices].astype(np.float32)

    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout)

    try:
        fold_briers = []
        for train_idx, test_idx in splits:
            model = make_model(model_type, hp)
            model.fit(X_sub[train_idx], y[train_idx])
            probs = model.predict_proba(X_sub[test_idx])[:, 1]
            fold_briers.append(brier_score_loss(y[test_idx], probs))
            del model
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float(np.mean(fold_briers))
    except _Timeout:
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        return 0.30  # timeout penalty
    except Exception:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        return 0.30


class Individual:
    def __init__(self, features=None, model_type=None, hp=None):
        if features is None:
            prob = TARGET_FEATURES / max(n_features, 1)
            self.features = [1 if random.random() < prob else 0 for _ in range(n_features)]
        else:
            self.features = list(features)
        self.model_type = model_type or random.choices(
            list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self.hp = hp or {
            'n_estimators': random.randint(100, 300),
            'max_depth': random.randint(4, 9),
            'learning_rate': 10 ** random.uniform(-2.0, -0.7),
            'subsample': random.uniform(0.6, 1.0),
            'colsample_bytree': random.uniform(0.4, 1.0),
        }
        self.brier = 1.0
        self.n_feat = sum(self.features)
        self._enforce_cap()

    def _enforce_cap(self):
        indices = [i for i, v in enumerate(self.features) if v]
        if len(indices) > MAX_FEATURES:
            for i in random.sample(indices, len(indices) - MAX_FEATURES):
                self.features[i] = 0
        self.n_feat = sum(self.features)

    def eval(self):
        self.brier = evaluate(self.features, self.model_type, self.hp)
        return self.brier

    def mutate(self, rate):
        for i in range(len(self.features)):
            if random.random() < rate:
                self.features[i] = 1 - self.features[i]
        if random.random() < 0.25:
            self.hp['n_estimators'] = max(50, min(300,
                self.hp['n_estimators'] + random.randint(-50, 50)))
        if random.random() < 0.25:
            self.hp['max_depth'] = max(2, min(10,
                self.hp['max_depth'] + random.randint(-2, 2)))
        if random.random() < 0.25:
            self.hp['learning_rate'] = max(0.001, min(0.3,
                self.hp['learning_rate'] * 10 ** random.uniform(-0.3, 0.3)))
        if random.random() < 0.08:
            self.model_type = random.choices(
                list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self._enforce_cap()
        self.brier = 1.0

    @staticmethod
    def crossover(p1, p2):
        point = random.randint(1, len(p1.features) - 1)
        child = Individual(
            features=p1.features[:point] + p2.features[point:],
            model_type=p1.model_type if random.random() < 0.5 else p2.model_type,
            hp={k: p1.hp[k] if random.random() < 0.5 else p2.hp[k] for k in p1.hp})
        return child


# ── Build or resume population ──
name_to_idx = {name: i for i, name in enumerate(feature_names)}

# State file lives in /kaggle/working/ -- persists within session.
# Download evolution_state_v2_kaggle.json from Kaggle output to resume later.
STATE_FILE = Path(f'{CACHE_DIR}/evolution_state_v2_kaggle.json')
population = []
best_ever_brier = 1.0
best_ever_info  = None
start_gen = 0

if STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text())
        start_gen       = state.get('generation', 0)
        best_ever_brier = state.get('best_brier', 1.0)
        best_ever_info  = state.get('best_info')
        for saved in state.get('population', []):
            feat = [0] * n_features
            for fname in saved.get('features', []):
                if fname in name_to_idx:
                    feat[name_to_idx[fname]] = 1
            ind = Individual(features=feat,
                             model_type=saved.get('model_type', 'tabicl'),
                             hp=saved.get('hp', {}))
            ind.brier = saved.get('brier', 1.0)
            population.append(ind)
        print(f'RESUMED: gen={start_gen}, best={best_ever_brier:.5f}, pop={len(population)}')
    except Exception as e:
        print(f'State load failed ({e}) -- starting fresh')

# Seed from islands if no saved state
if len(population) < POP_SIZE:
    for seed in island_seeds:
        if len(population) >= POP_SIZE:
            break
        feat = [0] * n_features
        for fname in seed.get('features', []):
            if isinstance(fname, str) and fname in name_to_idx:
                feat[name_to_idx[fname]] = 1
        if sum(feat) < 15:
            prob = TARGET_FEATURES / max(n_features, 1)
            feat = [1 if random.random() < prob else 0 for _ in range(n_features)]
        # TabICL variant of every seed (GPU evolution focus)
        population.append(Individual(features=feat, model_type='tabicl'))
        if len(population) < POP_SIZE:
            population.append(Individual(features=list(feat),
                                         model_type=seed.get('model_type', 'xgboost')))

    while len(population) < POP_SIZE:
        population.append(Individual())
    population = population[:POP_SIZE]

mt_counts = Counter(ind.model_type for ind in population)
print(f'Population ready: {len(population)} | Models: {dict(mt_counts)}')

In [ ]:
# ============================================================
# CELL 5: EVOLUTION LOOP (9-hour Kaggle session)
# ============================================================
# State saved to /kaggle/working/ every generation -- survives kernel restarts.
# Download evolution_state_v2_kaggle.json from Kaggle output to resume later.
# NOTE: /kaggle/working/ is NOT preserved across separate Kaggle sessions.
print('='*70)
print(f'  NBA QUANT AI -- GPU EVOLUTION v2 (Kaggle / {PLATFORM})')
print(f'  Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS} | Resume gen={start_gen}')
print(f'  Session limit: 9h | ATR to beat: 0.21837 (S13 CatBoost gen815)')
print('='*70)

mut_rate = MUTATION_RATE
session_start = time.time()
gens_this_session = 0
SESSION_LIMIT_H = 8.5  # stop 30min before Kaggle 9h hard kill


def save_state(gen, population, best_brier, best_info, mut_rate):
    """Checkpoint every generation. Download from Kaggle output to resume next session."""
    pop_data = []
    for ind in population:
        pop_data.append({
            'features': [feature_names[i] for i, v in enumerate(ind.features) if v],
            'model_type': ind.model_type,
            'hp': ind.hp,
            'brier': ind.brier,
        })
    STATE_FILE.write_text(json.dumps({
        'generation': gen + 1,
        'best_brier': best_brier,
        'best_info': best_info,
        'mutation_rate': mut_rate,
        'population': pop_data,
        'platform': PLATFORM,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, default=str))


for gen in range(start_gen, TOTAL_GENS):
    elapsed_h = (time.time() - session_start) / 3600
    if elapsed_h >= SESSION_LIMIT_H:
        print(f'\nSession limit reached ({elapsed_h:.1f}h). Stopping early.')
        print('Download evolution_state_v2_kaggle.json from Kaggle output to resume.')
        break

    gen_start = time.time()

    # Evaluate unevaluated individuals
    n_eval = 0
    for ind in population:
        if ind.brier >= 0.99:
            ind.eval()
            n_eval += 1

    population.sort(key=lambda x: x.brier)

    # Track best ever
    gen_best = population[0]
    improved = gen_best.brier < best_ever_brier
    if improved:
        best_ever_brier = gen_best.brier
        best_ever_info  = {
            'brier': gen_best.brier, 'model_type': gen_best.model_type,
            'n_features': gen_best.n_feat, 'generation': gen + 1,
            'features': [feature_names[i] for i, v in enumerate(gen_best.features) if v],
            'hp': gen_best.hp,
        }

    gen_dur   = time.time() - gen_start
    elapsed   = (time.time() - session_start) / 60
    gens_this_session += 1
    rate      = gens_this_session / max(elapsed, 0.1) * 60
    remaining = SESSION_LIMIT_H * 60 - elapsed
    est_left  = int(remaining / max(elapsed / max(gens_this_session, 1), 0.1))

    marker = ' *** NEW BEST ***' if improved else ''
    print(f'Gen {gen+1}/{TOTAL_GENS}: best={gen_best.brier:.5f} '
          f'({gen_best.model_type}, {gen_best.n_feat}f) '
          f'| ever={best_ever_brier:.5f} | {n_eval} evals {gen_dur:.0f}s '
          f'| {elapsed:.0f}min {rate:.0f}g/h ~{est_left}g left{marker}')

    if (gen + 1) % 10 == 0:
        mt   = Counter(ind.model_type for ind in population)
        top5 = [(f'{ind.brier:.5f}', ind.model_type, ind.n_feat) for ind in population[:5]]
        print(f'  Models: {dict(mt)} | Top5: {top5}')

    # Checkpoint every gen (fast write)
    save_state(gen, population, best_ever_brier, best_ever_info, mut_rate)

    # ── Selection + Reproduction ──
    elite = population[:ELITE_SIZE]

    def tournament(pop, k=4):
        return min(random.sample(pop, min(k, len(pop))), key=lambda x: x.brier)

    children = []
    for e in elite:
        c = Individual(features=list(e.features), model_type=e.model_type, hp=dict(e.hp))
        c.brier = e.brier
        children.append(c)

    while len(children) < POP_SIZE:
        if random.random() < CROSSOVER_RATE:
            child = Individual.crossover(tournament(population), tournament(population))
        else:
            p = tournament(population)
            child = Individual(features=list(p.features), model_type=p.model_type,
                               hp=dict(p.hp))
        child.mutate(mut_rate)
        # Force TabICL in 40% of new children (GPU evolution focus)
        if random.random() < 0.40:
            child.model_type = 'tabicl'
        children.append(child)

    population = children[:POP_SIZE]
    mut_rate = max(MUT_FLOOR, min(0.15, mut_rate * MUT_DECAY))


# Final checkpoint
save_state(TOTAL_GENS - 1, population, best_ever_brier, best_ever_info, mut_rate)
print(f'\nDone! {gens_this_session} gens in {(time.time()-session_start)/60:.0f} min')
print(f'State file: {STATE_FILE}')
print('-> Download evolution_state_v2_kaggle.json from Kaggle output to resume next session.')

In [ ]:
# ============================================================
# CELL 6: RESULTS + SAVE FOR HF ISLAND INJECTION
# ============================================================
# Kaggle GPU sessions have no internet -- HF island injection will fail here.
# Best features are saved to /kaggle/working/ (appears in Kaggle output).
# After session: download and inject from your VM (which has internet).
import json, time, requests
from collections import Counter
from pathlib import Path

CACHE_DIR = '/kaggle/working/nba-quant-gpu'

print('\n' + '='*70)
print('  FINAL RESULTS')
print('='*70)

if best_ever_info:
    print(f'  Best Brier:    {best_ever_info["brier"]:.5f}')
    print(f'  Model:         {best_ever_info["model_type"]}')
    print(f'  Features:      {best_ever_info["n_features"]}')
    print(f'  Generation:    {best_ever_info["generation"]}')
    print(f'  Session gens:  {gens_this_session}')
    print(f'  Session time:  {(time.time()-session_start)/60:.0f} min')

    population.sort(key=lambda x: x.brier)
    print(f'\n  Top 10:')
    for i, ind in enumerate(population[:10]):
        print(f'    #{i+1}: brier={ind.brier:.5f} | {ind.model_type} | {ind.n_feat}f')

    mt_top = Counter(ind.model_type for ind in population[:20])
    print(f'\n  Models in top 20: {dict(mt_top)}')

    print(f'\n  Current ATR:  0.21837 (S13 CatBoost)')
    delta = best_ever_info['brier'] - 0.21837
    print(f'  Delta vs ATR: {delta:+.5f} {"*** NEW RECORD! ***" if delta < 0 else ""}')

    # Save best features to Kaggle output for manual injection
    best_features_file = Path(f'{CACHE_DIR}/best_gpu_features_kaggle.json')
    best_features_file.write_text(json.dumps({
        'brier': best_ever_info['brier'],
        'model_type': best_ever_info['model_type'],
        'features': best_ever_info['features'],
        'n_features': best_ever_info['n_features'],
        'generation': best_ever_info['generation'],
        'source': f'gpu_v2_{PLATFORM}',
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, indent=2))
    print(f'\n  Best features saved to: {best_features_file}')

    # Attempt HF injection -- fails silently in GPU mode (no internet)
    print('\n  Attempting HF island injection (no-op in GPU mode -- no internet)...')
    ISLANDS = {
        'S10': 'https://nomos42-nba-quant.hf.space',
        'S11': 'https://nomos42-nba-quant-2.hf.space',
    }
    payload = {
        'seed_features': best_ever_info['features'],
        'seed_model_type': best_ever_info['model_type'],
        'seed_brier': best_ever_info['brier'],
        'source': f'kaggle_gpu_v2_gen{best_ever_info["generation"]}',
    }
    for name, url in ISLANDS.items():
        try:
            resp = requests.post(f'{url}/api/config', json=payload, timeout=5)
            if resp.status_code == 200:
                print(f'  {name}: injected OK')
            else:
                print(f'  {name}: HTTP {resp.status_code} -- inject manually from VM')
        except Exception as e:
            print(f'  {name}: {e} (expected -- no internet in GPU mode)')

    print('\n  NEXT STEPS:')
    print('  1. Download from Kaggle output:')
    print('       best_gpu_features_kaggle.json    (features to inject into HF islands)')
    print('       evolution_state_v2_kaggle.json   (resume state for next session)')
    print('  2. Inject into HF islands from VM (internet available):')
    print('       curl -X POST https://nomos42-nba-quant.hf.space/api/config \\')
    print('            -H "Content-Type: application/json" \\')
    print('            -d @best_gpu_features_kaggle.json')
    print('  3. To resume next session: upload evolution_state_v2_kaggle.json,')
    print(f'     copy to /kaggle/working/nba-quant-gpu/ before running Cell 4.')
else:
    print('  No results yet -- run Cell 5 first.')